# Text Preprocessing — Notebook
### AI Engineering Course

This notebook walks through every topic in the Text Preprocessing lecture, with live code demonstrations for each concept.

**Topics covered:**
1. Why preprocess text?
2. Cleaning & normalization
3. Vocabulary operations
4. Word-level tokenization
5. Subword tokenization
6. From text to numbers: BoW, TF-IDF, BM25

In [ ]:
# Install dependencies (run once)
# !pip install nltk spacy rank-bm25 scikit-learn transformers beautifulsoup4
# !python -m spacy download en_core_web_sm

import re
import unicodedata
import warnings
warnings.filterwarnings('ignore')

print('Imports ready')

---
## Part 1 — Why preprocess text?

Computers cannot work with raw text directly — they only understand numbers.  
Text is inconsistent, noisy, ambiguous, and variable in length.  
Preprocessing transforms raw text into a clean, standardized form.

In [ ]:
# The problem with raw text
examples = [
    "Dog", "dog", "DOG",          # same word, different cases
    "<p>Hello World</p>",          # HTML noise
    "running", "runs", "ran",      # same root, different forms
    "U.S.A. is great",             # abbreviations
    "the", "a", "is",              # stop words — low information
]

print("Raw text examples that a model sees as unrelated:")
for e in examples:
    print(f"  {repr(e)}")

print()
print("After preprocessing, many of these should collapse to the same token.")

In [ ]:
# Quick illustration: the effect of preprocessing
raw_pairs = [
    ("The dog jumped!!", "dog jump"),
    ("<p>Hello World</p>", "hello world"),
    ("U.S.A. is great", "usa great"),
    ("running, runs, ran", "run run run"),
]

print(f"{'Raw text':<30} {'After preprocessing':<25}")
print("-" * 55)
for raw, clean in raw_pairs:
    print(f"{raw:<30} {clean:<25}")

---
## Part 2 — Cleaning & Normalization

### 2.1 Lowercasing

In [ ]:
text = "The Quick Brown Fox Jumps Over The Lazy Dog"
lowercased = text.lower()

print(f"Original : {text}")
print(f"Lowercased: {lowercased}")

# Vocabulary size comparison
tokens_original = set(text.split())
tokens_lower = set(lowercased.split())
print(f"\nUnique tokens before: {len(tokens_original)}")
print(f"Unique tokens after:  {len(tokens_lower)}")

# The exception: case matters sometimes
print("\n⚠️  Case can matter:")
print("  'US' (country) vs 'us' (pronoun)")
print("  'Apple' (company) vs 'apple' (fruit)")

### 2.2 Removing punctuation and special characters

In [ ]:
text = "Hello, World! This is NLP. Isn't it great? Cost: $99.99"

# Remove all non-word, non-space characters
cleaned = re.sub(r'[^\w\s]', '', text)
print(f"Original : {text}")
print(f"No punct : {cleaned}")

print()
# Sometimes we want to keep certain punctuation
# e.g. hyphens in compound words, apostrophes in contractions
text2 = "state-of-the-art doesn't mean anything without context"
cleaned2 = re.sub(r'[^\w\s\-\']', '', text2)
print(f"Original : {text2}")
print(f"Kept - ' : {cleaned2}")

### 2.3 Removing HTML, URLs, and noise

In [ ]:
# --- HTML tags ---
html_text = "<h1>Product Review</h1><p>This <strong>laptop</strong> is <em>amazing</em>!</p>"
no_html = re.sub(r'<[^>]+>', '', html_text)
print("HTML removal:")
print(f"  Before: {html_text}")
print(f"  After : {no_html}")

In [ ]:
# For real HTML, BeautifulSoup is more robust than regex
from bs4 import BeautifulSoup

html = "<div><h2>Best Laptops 2024</h2><p>Check <a href='http://example.com'>this link</a>.</p></div>"
text = BeautifulSoup(html, 'html.parser').get_text(separator=' ')
print("BeautifulSoup:")
print(f"  Before: {html}")
print(f"  After : {text.strip()}")

In [ ]:
# --- URLs and emails ---
text = "Check out our blog at https://myblog.com/post/123 or www.example.com. Contact: info@company.co.il"
print(f"Original: {text}")

text = re.sub(r'http\S+|www\.\S+', '', text)
print(f"No URLs : {text}")

text = re.sub(r'\S+@\S+', '', text)
print(f"No email: {text}")

text = re.sub(r'\s+', ' ', text).strip()
print(f"Cleaned : {text}")

In [ ]:
# --- Full pipeline on a scraped snippet ---
def remove_noise(text):
    text = re.sub(r'<[^>]+>', ' ', text)       # strip HTML
    text = re.sub(r'http\S+|www\.\S+', '', text) # strip URLs
    text = re.sub(r'\S+@\S+', '', text)          # strip emails
    text = re.sub(r'\s+', ' ', text).strip()     # normalize whitespace
    return text

raw = '<div class="post"><h2>Best Laptops 2024</h2><p>Visit https://techreview.com for details. Contact: editor@techreview.com</p></div>'
print(f"Raw   : {raw}")
print(f"Cleaned: {remove_noise(raw)}")

### 2.4 Handling numbers — three strategies

In [ ]:
text = "On January 5th 2024, the temperature was -3 degrees and 12 people attended"

# Strategy 1: remove entirely
s1 = re.sub(r'\d+', '', text)
s1 = re.sub(r'\s+', ' ', s1).strip()
print(f"Original        : {text}")
print(f"Strategy 1 (rm) : {s1}")

# Strategy 2: replace with placeholder
s2 = re.sub(r'\d+\.\d+', '<FLOAT>', text)
s2 = re.sub(r'\d+', '<INT>', s2)
print(f"Strategy 2 (<INT>): {s2}")

# Strategy 3: keep as-is, just normalize formatting
text_financial = "Revenue was $1,234,567 in Q3, up 12.5%"
s3 = text_financial.replace(',', '').replace('$', '').replace('%', ' percent')
print(f"\nFinancial original: {text_financial}")
print(f"Strategy 3 (keep) : {s3}")

In [ ]:
# When to use which strategy
strategies = {
    "Remove entirely": "Sentiment analysis — '5-star hotel was amazing' → star rating redundant",
    "Replace <NUM>": "NER — 'room 42' → model learns '<NUM>' is a location pattern",
    "Keep as-is": "Finance — 'stock fell 12%' vs 'stock fell 40%' — magnitude matters",
}
for strategy, example in strategies.items():
    print(f"{strategy:>20}: {example}")

### 2.5 Unicode normalization

In [ ]:
import unicodedata

# The same character encoded two different ways
e_composed   = '\u00e9'          # é as a single codepoint (NFC)
e_decomposed = 'e\u0301'         # é as e + combining accent (NFD)

print(f"Composed   : {e_composed!r}  → renders as: {e_composed}")
print(f"Decomposed : {e_decomposed!r} → renders as: {e_decomposed}")
print(f"They look the same: {e_composed == e_decomposed}")  # False!
print()

# NFC: normalize to composed form — one canonical form for storage/comparison
normalized = unicodedata.normalize('NFC', e_decomposed)
print(f"After NFC: {normalized!r}")
print(f"Now equal: {e_composed == normalized}")

In [ ]:
# Stripping accents: NFD decompose, then drop combining marks (category 'Mn')
text = "café résumé naïve Ångström"

def strip_accents(text):
    text = unicodedata.normalize('NFD', text)            # decompose
    text = ''.join(c for c in text                       # drop Mark, Nonspacing
                   if unicodedata.category(c) != 'Mn')
    return text

print(f"Original      : {text}")
print(f"Accents stripped: {strip_accents(text)}")

In [ ]:
# Other invisible Unicode noise from PDFs and Word docs
messy = "\u201cHello\u201d he said\u2014 and left. Price\u00a0was\u00a0$10."
print(f"Messy  : {messy!r}")
print(f"Display: {messy}")

def normalize_unicode(text):
    text = text.replace('\u201c', '"').replace('\u201d', '"')  # smart quotes
    text = text.replace('\u2018', "'").replace('\u2019', "'")
    text = text.replace('\u2014', '-').replace('\u2013', '-')  # em/en dash
    text = text.replace('\u00a0', ' ')                          # non-breaking space
    text = unicodedata.normalize('NFC', text)
    return text

clean = normalize_unicode(messy)
print(f"\nCleaned: {clean!r}")
print(f"Display: {clean}")

### 2.6 The full cleaning pipeline

In [ ]:
def clean_text(text, remove_numbers=True):
    """Full text cleaning pipeline."""
    # 1. Normalize unicode
    text = normalize_unicode(text)
    # 2. Strip HTML
    text = re.sub(r'<[^>]+>', ' ', text)
    # 3. Strip URLs and emails
    text = re.sub(r'http\S+|www\.\S+', '', text)
    text = re.sub(r'\S+@\S+', '', text)
    # 4. Lowercase
    text = text.lower()
    # 5. Handle numbers
    if remove_numbers:
        text = re.sub(r'\d+', '', text)
    # 6. Remove punctuation
    text = re.sub(r'[^\w\s]', '', text)
    # 7. Strip extra whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    return text

sample = '<p>Check out https://example.com! <strong>Amazing</strong> deal\u2014only $29.99 on Jan 5th 2024.</p>'
print(f"Raw    : {sample}")
print(f"Cleaned: {clean_text(sample)}")

---
## Part 3 — Vocabulary Operations

### 3.1 Stop words

In [ ]:
import nltk
nltk.download('stopwords', quiet=True)
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
from nltk.corpus import stopwords

stop_words = set(stopwords.words('english'))
print(f"Number of English stop words: {len(stop_words)}")
print(f"Sample: {sorted(list(stop_words))[:20]}")

In [ ]:
text = "The cat sat on the mat and looked at the window"
tokens = text.lower().split()

filtered = [t for t in tokens if t not in stop_words]

print(f"Original tokens : {tokens}")
print(f"Without stops   : {filtered}")
print(f"Tokens removed  : {[t for t in tokens if t in stop_words]}")

print()
# Warning: 'not' is a stop word — can break sentiment analysis
text2 = "this movie is not good at all"
tokens2 = text2.split()
filtered2 = [t for t in tokens2 if t not in stop_words]
print(f"⚠️  Sentiment risk:")
print(f"  Original : {tokens2}")
print(f"  Filtered : {filtered2}  ← 'not' removed, meaning flipped!")

### 3.2 Stemming

In [ ]:
from nltk.stem import PorterStemmer, SnowballStemmer
from nltk.stem.regexp import RegexpStemmer

words = ["running", "runner", "runs", "ran",
         "studies", "studying", "studied",
         "easily", "fairly", "happiness"]

porter = PorterStemmer()
snowball = SnowballStemmer('english')
regexp = RegexpStemmer('ing$|s$|ed$', min=4)  # simple custom rules

print(f"{'Word':<15} {'Porter':<15} {'Snowball':<15} {'Regexp':<15}")
print("-" * 60)
for word in words:
    print(f"{word:<15} {porter.stem(word):<15} {snowball.stem(word):<15} {regexp.stem(word):<15}")

In [ ]:
# Snowball supports multiple languages
print("Snowball supported languages:", SnowballStemmer.languages)

# Example: French stemming
fr_stemmer = SnowballStemmer('french')
french_words = ["chanter", "chanteur", "chantons", "mangeons", "manger"]
print()
print("French stemming:")
for w in french_words:
    print(f"  {w} → {fr_stemmer.stem(w)}")

### 3.3 Lemmatization

In [ ]:
import spacy
nlp = spacy.load('en_core_web_sm')

text = "The children were running quickly and the geese flew better than before"
doc = nlp(text)

print(f"{'Token':<15} {'Lemma':<15} {'POS':<10}")
print("-" * 40)
for token in doc:
    print(f"{token.text:<15} {token.lemma_:<15} {token.pos_:<10}")

In [ ]:
# Stemming vs Lemmatization: direct comparison
words = ["better", "was", "children", "geese", "running", "studies"]

print(f"{'Word':<12} {'Porter stem':<15} {'Lemma (spaCy)':<15}")
print("-" * 42)
for word in words:
    doc = nlp(word)
    lemma = doc[0].lemma_
    stem = porter.stem(word)
    print(f"{word:<12} {stem:<15} {lemma:<15}")

print()
print("Key differences:")
print("  'better' → stemmer keeps it, lemmatizer gives 'good'")
print("  'was'    → stemmer gives 'wa', lemmatizer gives 'be'")
print("  'geese'  → stemmer gives 'gees', lemmatizer gives 'goose'")

### 3.4 N-grams

In [ ]:
from nltk import ngrams

tokens = ["I", "love", "natural", "language", "processing"]

for n in [1, 2, 3]:
    name = {1: "Unigrams", 2: "Bigrams", 3: "Trigrams"}[n]
    grams = list(ngrams(tokens, n))
    print(f"{name}: {grams}")

In [ ]:
# Why bigrams matter: negation and compound concepts
examples = [
    ("not good", "bigram captures negation; unigram loses it"),
    ("New York", "two words, one concept"),
    ("machine learning", "compound concept split by unigrams"),
]

for phrase, reason in examples:
    words = phrase.split()
    unigrams = words
    bigram = [" ".join(words)]
    print(f"Phrase: '{phrase}'")
    print(f"  Unigrams: {unigrams}")
    print(f"  Bigram  : {bigram}  ← {reason}")
    print()

---
## Part 4 — Word-level Tokenization

### 4.1 What is a token?

In [ ]:
# Simple whitespace tokenization
text = "I love NLP"
tokens = text.split()
print(f"Text  : '{text}'")
print(f"Tokens: {tokens}")
print(f"Count : {len(tokens)} tokens")

### 4.2 Building a vocabulary and assigning token IDs

In [ ]:
corpus = [
    "The quick brown fox",
    "The fox jumped over the lazy dog",
    "Dogs and foxes are animals",
]

# Tokenize and build vocabulary
all_tokens = []
for doc in corpus:
    all_tokens.extend(doc.lower().split())

vocab = sorted(set(all_tokens))  # alphabetical order
word2id = {word: idx for idx, word in enumerate(vocab)}
id2word = {idx: word for word, idx in word2id.items()}

print("Vocabulary (word → ID):")
for word, idx in word2id.items():
    print(f"  {word:<12} → {idx}")

In [ ]:
# Encode and decode
def encode(text, word2id, unk_id):
    return [word2id.get(t, unk_id) for t in text.lower().split()]

def decode(ids, id2word):
    return [id2word.get(i, '<UNK>') for i in ids]

UNK_ID = len(word2id)  # next available ID

test = "the quick fox jumped"
encoded = encode(test, word2id, UNK_ID)
decoded = decode(encoded, id2word)

print(f"Text   : {test}")
print(f"Encoded: {encoded}")
print(f"Decoded: {decoded}")

### 4.3 Special tokens

In [ ]:
# Special tokens extend the vocabulary
special_tokens = {
    '<UNK>': 'Unknown word — not in vocabulary',
    '<PAD>': 'Padding — make sequences equal length',
    '<BOS>': 'Beginning of sequence',
    '<EOS>': 'End of sequence',
    '<CLS>': 'Classification token (BERT-style)',
    '<SEP>': 'Separator between segments (BERT-style)',
    '<MASK>': 'Masked token for MLM pre-training',
    '<|endoftext|>': 'GPT-style end-of-document marker',
}

print(f"{'Token':<20} {'Purpose'}")
print("-" * 60)
for token, purpose in special_tokens.items():
    print(f"{token:<20} {purpose}")

In [ ]:
# Padding example: making variable-length sequences equal
sequences = [
    [4, 7, 2],
    [1, 9],
    [3, 5, 8, 6, 2],
]

PAD_ID = 0
max_len = max(len(s) for s in sequences)
padded = [s + [PAD_ID] * (max_len - len(s)) for s in sequences]

print("Before padding:", sequences)
print("After padding :", padded)
print(f"All length {max_len} — ready for batch processing")

### 4.4 Word tokenization challenges

In [ ]:
# NLTK word tokenizer handles many edge cases
nltk.download('punkt', quiet=True)
from nltk.tokenize import word_tokenize

challenges = [
    "don't can't won't",           # contractions
    "state-of-the-art",             # hyphenated
    "end.",                         # punctuation attached
    "U.S.A. is great.",             # abbreviations with periods
    "$100.50 or €200",              # currency
]

print(f"{'Input':<30} {'Tokens'}")
print("-" * 70)
for text in challenges:
    tokens = word_tokenize(text)
    print(f"{text:<30} {tokens}")

### 4.5 Sentence tokenization

In [ ]:
from nltk.tokenize import sent_tokenize

# The tricky cases — periods inside sentences
text = "Dr. Smith went to Washington D.C. He loved it. The U.S. govt paid $1.5M."
sentences = sent_tokenize(text)

print(f"Input: {text}")
print()
print("Sentences detected:")
for i, s in enumerate(sentences, 1):
    print(f"  {i}. {s}")

### 4.6 Character-level tokenization

In [ ]:
word = "hello"
char_tokens = list(word)
print(f"Word: '{word}'")
print(f"Character tokens: {char_tokens}")
print(f"Vocabulary size: just {len(set(char_tokens))} unique characters")

print()
# Advantage: handles ANY word, including misspellings
words = ["hello", "helo", "h3ll0", "superkalafragalisticexpialidocious"]
for w in words:
    chars = list(w)
    print(f"  '{w}' → {chars[:8]}{'...' if len(chars) > 8 else ''}  ({len(chars)} chars)")

print()
print("Trade-off: sequences become very long — 'hello' is 5 tokens instead of 1")

---
## Part 5 — Subword Tokenization

### 5.1 The OOV problem with word-level tokenizers

In [ ]:
# Simulate a word-level tokenizer with a fixed training vocabulary
training_vocab = {"the", "cat", "sat", "on", "mat", "dog", "chased", "happy", "sad"}
UNK = "<UNK>"

def word_tokenize_with_unk(text, vocab):
    return [t if t in vocab else UNK for t in text.lower().split()]

# Inference-time sentences with unseen words
test_sentences = [
    "the cat sat on the mat",            # all known
    "the cryptocurrency crashed overnight", # 'cryptocurrency' unknown
    "dogecoin blockchain nft crashed",      # all financial jargon = all UNK
]

for sent in test_sentences:
    tokens = word_tokenize_with_unk(sent, training_vocab)
    print(f"Input  : {sent}")
    print(f"Tokens : {tokens}")
    unk_count = tokens.count(UNK)
    if unk_count > 0:
        print(f"⚠️  {unk_count} word(s) lost to <UNK>")
    print()

### 5.2 Subword tokenization with a real tokenizer (BPE — GPT-2)

In [ ]:
from transformers import GPT2Tokenizer, BertTokenizer

gpt2_tokenizer = GPT2Tokenizer.from_pretrained('gpt2')
bert_tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

print(f"GPT-2 vocabulary size : {gpt2_tokenizer.vocab_size:,}")
print(f"BERT vocabulary size  : {bert_tokenizer.vocab_size:,}")

In [ ]:
# BPE (GPT-2) — how it handles various words
words = [
    "tokenization",
    "unhappiness",
    "cryptocurrency",
    "Akwirwier",           # completely made-up word
    "1234",
    "12345",
    "12346",
]

print("GPT-2 BPE tokenization:")
print(f"{'Word':<20} {'Tokens':<40} {'Count'}")
print("-" * 65)
for word in words:
    token_ids = gpt2_tokenizer.encode(word)
    token_strs = gpt2_tokenizer.convert_ids_to_tokens(token_ids)
    print(f"{word:<20} {str(token_strs):<40} {len(token_strs)}")

In [ ]:
# BERT WordPiece — note the ## prefix on continuation subwords
print("BERT WordPiece tokenization:")
print(f"{'Word':<20} {'Tokens':<40} {'Count'}")
print("-" * 65)
for word in words:
    tokens = bert_tokenizer.tokenize(word)
    print(f"{word:<20} {str(tokens):<40} {len(tokens)}")

print()
print("Note: ## prefix = continuation subword (not word-initial)")

In [ ]:
# Numbers: inconsistent tokenization — the core problem
print("Number tokenization (GPT-2) — notice the arbitrary splits:")
numbers = [str(n) for n in [1, 12, 123, 1234, 12345, 12346, 99, 100, 101, 1000, 1001]]
for n in numbers:
    tokens = gpt2_tokenizer.convert_ids_to_tokens(gpt2_tokenizer.encode(n))
    print(f"  {n:<8} → {tokens}")

print()
print("Consequence: '1000' and '1001' may have completely different subword representations.")
print("The model has no built-in sense of numeric proximity or magnitude.")

### 5.3 BPE training algorithm (step by step)

In [ ]:
from collections import Counter, defaultdict

def get_pairs(vocab):
    """Count all adjacent symbol pairs in the vocabulary."""
    pairs = Counter()
    for word, freq in vocab.items():
        symbols = word.split()
        for i in range(len(symbols) - 1):
            pairs[(symbols[i], symbols[i+1])] += freq
    return pairs

def merge_pair(pair, vocab):
    """Merge all occurrences of a pair into a single symbol."""
    new_vocab = {}
    bigram = ' '.join(pair)
    replacement = ''.join(pair)
    for word, freq in vocab.items():
        new_word = word.replace(bigram, replacement)
        new_vocab[new_word] = freq
    return new_vocab

# Small training corpus — each word split into chars, with </w> marking word-end
vocab = {
    'l o w </w>': 5,
    'l o w e r </w>': 2,
    'n e w e s t </w>': 6,
    'w i d e s t </w>': 3,
}

print("Initial vocabulary:")
for word, freq in vocab.items():
    print(f"  {word!r:30} freq={freq}")

print()
num_merges = 8
merges = []

for i in range(num_merges):
    pairs = get_pairs(vocab)
    if not pairs:
        break
    best = max(pairs, key=pairs.get)
    vocab = merge_pair(best, vocab)
    merges.append(best)
    print(f"Merge {i+1}: {best} → {''.join(best)}  (freq={pairs[best]})")

print()
print("Final vocabulary:")
for word in vocab:
    print(f"  {word}")

### 5.4 SentencePiece — language-agnostic tokenization

In [ ]:
# SentencePiece is used in T5, LLaMA, mBART
# Demonstrate with a T5 tokenizer which uses SentencePiece
from transformers import T5Tokenizer

t5_tokenizer = T5Tokenizer.from_pretrained('t5-small')

texts = [
    "Hello world",
    "tokenization is fascinating",
    "שלום עולם",    # Hebrew
    "日本語のテスト",  # Japanese
]

print("SentencePiece tokenization (T5):")
print("Note: ▁ marks the start of a new word (space absorbed into token)")
print()
for text in texts:
    tokens = t5_tokenizer.tokenize(text)
    print(f"  '{text}'")
    print(f"  → {tokens}")
    print()

---
## Part 6 — From Text to Numbers

### 6.1 Bag of Words (BoW)

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
import numpy as np

documents = [
    "I love NLP",
    "I love machine learning",
    "NLP is machine learning",
]

vectorizer = CountVectorizer()
bow_matrix = vectorizer.fit_transform(documents)
vocab = vectorizer.get_feature_names_out()

print("Vocabulary:", list(vocab))
print()

import pandas as pd
df = pd.DataFrame(bow_matrix.toarray(), columns=vocab,
                  index=[f'D{i+1}' for i in range(len(documents))])
print(df)

In [ ]:
# BoW loses word order — same vector for different meanings
sentences = [
    "dog bites man",
    "man bites dog",
]

v = CountVectorizer()
m = v.fit_transform(sentences)
print("BoW cannot distinguish word order:")
for i, sent in enumerate(sentences):
    print(f"  '{sent}' → {m.toarray()[i]}")
print(f"Vocab: {v.get_feature_names_out()}")
print("Vectors are identical — the model sees these as the same!")

### 6.2 TF-IDF

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

corpus = [
    "the cat sat on the mat",
    "the dog chased the cat around the mat",
    "neural networks learn representations of data",
    "deep learning uses neural networks with many layers",
]

tfidf = TfidfVectorizer()
matrix = tfidf.fit_transform(corpus)
vocab = tfidf.get_feature_names_out()

df_tfidf = pd.DataFrame(
    matrix.toarray().round(3),
    columns=vocab,
    index=[f'D{i+1}' for i in range(len(corpus))]
)

print("TF-IDF matrix (selected columns):")
# Show interesting words
interesting = ['cat', 'dog', 'mat', 'the', 'neural', 'learning', 'networks']
interesting = [w for w in interesting if w in df_tfidf.columns]
print(df_tfidf[interesting].to_string())

In [ ]:
# 'the' gets low TF-IDF because it appears in many documents
# 'neural', 'cat' get higher scores because they are distinctive
print("IDF values (higher = rarer across documents):")
idf_scores = dict(zip(vocab, tfidf.idf_))
for word in sorted(idf_scores, key=idf_scores.get)[:10]:
    print(f"  {word:<15} IDF = {idf_scores[word]:.3f}")

print("  ...")
for word in sorted(idf_scores, key=idf_scores.get, reverse=True)[:5]:
    print(f"  {word:<15} IDF = {idf_scores[word]:.3f}")

In [ ]:
# Using TF-IDF for document similarity
from sklearn.metrics.pairwise import cosine_similarity

query = ["cat on the mat"]
query_vec = tfidf.transform(query)

similarities = cosine_similarity(query_vec, matrix)[0]

print(f"Query: '{query[0]}'")
print()
print("Similarity to each document:")
for i, (doc, score) in enumerate(zip(corpus, similarities)):
    print(f"  D{i+1} ({score:.3f}): {doc}")

### 6.3 BM25 — Best Match 25

BM25 improves on TF-IDF with two key mechanisms:
- **TF saturation**: term frequency contributions plateau — the 50th occurrence adds far less than the 1st
- **Document length normalization**: longer documents are penalized to avoid length bias

In [ ]:
# TF saturation: BM25 vs raw TF
import numpy as np

k1 = 1.5  # typical value

tf_values = [1, 2, 5, 10, 20, 50, 100]

print(f"{'TF (occurrences)':<20} {'Raw TF score':<18} {'BM25 TF component (k1={k1})'}")
print("-" * 60)
for tf in tf_values:
    bm25_tf = (tf * (k1 + 1)) / (tf + k1)
    print(f"{tf:<20} {tf:<18} {bm25_tf:.3f}")

print()
print(f"BM25 asymptote: k1 + 1 = {k1 + 1}  ← TF can never exceed this")

In [ ]:
import matplotlib.pyplot as plt

tf_range = np.linspace(0, 50, 300)
k1 = 1.5

raw_tf = tf_range
bm25_tf = (tf_range * (k1 + 1)) / (tf_range + k1)

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(tf_range, raw_tf, label='Raw TF (used in TF-IDF)', linewidth=2)
ax.plot(tf_range, bm25_tf, label=f'BM25 TF component (k₁={k1})', linewidth=2)
ax.axhline(k1 + 1, color='gray', linestyle='--', alpha=0.7, label=f'BM25 ceiling (k₁+1={k1+1})')
ax.set_xlabel('Term frequency (number of occurrences)', fontsize=12)
ax.set_ylabel('Score contribution', fontsize=12)
ax.set_title('TF saturation: BM25 vs raw TF', fontsize=13)
ax.legend(fontsize=11)
ax.set_ylim(0, 20)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
from rank_bm25 import BM25Okapi

corpus = [
    "the cat sat on the mat",
    "the dog chased the cat around the garden",
    "neural networks learn representations of data",
    "deep learning uses neural networks with many layers",
    "the cat and dog are domestic animals that people love",
]

tokenized_corpus = [doc.lower().split() for doc in corpus]
bm25 = BM25Okapi(tokenized_corpus)

query = "cat"
scores = bm25.get_scores(query.lower().split())

print(f"Query: '{query}'")
print()
ranked = sorted(enumerate(scores), key=lambda x: x[1], reverse=True)
for rank, (idx, score) in enumerate(ranked, 1):
    print(f"  Rank {rank} (score={score:.3f}): {corpus[idx]}")

In [ ]:
# Multi-term query
query = "cat dog"
scores = bm25.get_scores(query.lower().split())

print(f"Query: '{query}'")
print()
ranked = sorted(enumerate(scores), key=lambda x: x[1], reverse=True)
for rank, (idx, score) in enumerate(ranked, 1):
    print(f"  Rank {rank} (score={score:.3f}): {corpus[idx]}")

In [ ]:
# BM25 vs TF-IDF: head-to-head on same corpus
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

tfidf = TfidfVectorizer()
tfidf_matrix = tfidf.fit_transform(corpus)

query_vec = tfidf.transform(["cat dog"])
tfidf_scores = cosine_similarity(query_vec, tfidf_matrix)[0]

bm25_scores = bm25.get_scores("cat dog".split())

print(f"{'Document':<52} {'TF-IDF':>8} {'BM25':>8}")
print("-" * 70)
for i, doc in enumerate(corpus):
    short = doc[:50]
    print(f"{short:<52} {tfidf_scores[i]:>8.3f} {bm25_scores[i]:>8.3f}")

In [ ]:
# BM25 parameters: effect of k1 and b
print("Effect of k1 (TF saturation speed):")
print(f"  k1=0.5 → saturates very fast (almost binary: present/absent)")
print(f"  k1=1.2 → moderate saturation (common default)")
print(f"  k1=2.0 → slower saturation (rewards repeated terms more)")
print()
print("Effect of b (length normalization):")
print(f"  b=0.0  → no length normalization (long docs always win)")
print(f"  b=0.75 → partial normalization (common default)")
print(f"  b=1.0  → full normalization (score purely by density, not count)")
print()

query = "cat"
for k1 in [0.5, 1.2, 2.0]:
    bm = BM25Okapi(tokenized_corpus, k1=k1, b=0.75)
    scores = bm.get_scores(query.split())
    best_idx = int(np.argmax(scores))
    print(f"  k1={k1}: top doc → '{corpus[best_idx][:50]}' (score={scores[best_idx]:.3f})")

### 6.4 BoW / TF-IDF / BM25 limitations — and what comes next

In [ ]:
# Limitation 1: no word order
corpus_order = ["dog bites man", "man bites dog"]
v = CountVectorizer()
m = v.fit_transform(corpus_order).toarray()
print("No word order (BoW):")
print(f"  '{corpus_order[0]}' → {m[0]}")
print(f"  '{corpus_order[1]}' → {m[1]}")
print(f"  Identical? {np.array_equal(m[0], m[1])}")
print()

# Limitation 2: no semantics
tfidf2 = TfidfVectorizer()
vecs = tfidf2.fit_transform(["happy", "joyful", "sad"]).toarray()
sim_happy_joyful = cosine_similarity([vecs[0]], [vecs[1]])[0][0]
sim_happy_sad = cosine_similarity([vecs[0]], [vecs[2]])[0][0]
print("No semantics (TF-IDF):")
print(f"  cosine('happy', 'joyful') = {sim_happy_joyful:.3f}  ← synonyms look unrelated")
print(f"  cosine('happy', 'sad')    = {sim_happy_sad:.3f}  ← antonyms also unrelated")
print()

# Limitation 3: sparsity
tfidf3 = TfidfVectorizer()
matrix3 = tfidf3.fit_transform(corpus)
density = matrix3.nnz / (matrix3.shape[0] * matrix3.shape[1])
print(f"Sparsity:")
print(f"  Matrix shape: {matrix3.shape}")
print(f"  Non-zero entries: {matrix3.nnz} out of {matrix3.shape[0] * matrix3.shape[1]}")
print(f"  Density: {density:.1%}  ← {100 - density*100:.1f}% zeros")

In [ ]:
# Summary: when to use what
print("Method comparison:")
print()
methods = [
    ("BoW",     "Simple count",              "Baseline text classification, simple search"),
    ("TF-IDF",  "Weighted by rarity",        "Document retrieval, keyword extraction"),
    ("BM25",    "Saturated TF + length norm","Production search engines, RAG retrieval"),
    ("Embeddings", "Dense semantic vectors", "Semantic search, similarity, next lecture"),
]
print(f"{'Method':<12} {'Key idea':<30} {'Best use case'}")
print("-" * 80)
for method, idea, use in methods:
    print(f"{method:<12} {idea:<30} {use}")

---
## Part 7 — Classification comparison: BoW vs TF-IDF vs BM25

**Task:** 20 Newsgroups — classify news posts into 4 topic categories.

We compare three approaches:
1. **BoW + Multinomial Naive Bayes** — the classic baseline
2. **TF-IDF + Logistic Regression** — TF-IDF as features
3. **BM25 + Nearest-neighbor vote** — retrieval-based classifier

For BM25: each test document is scored against all training documents,
scores are summed per class, and the document is assigned to the highest-scoring class.

In [ ]:
from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report
from rank_bm25 import BM25Okapi
import numpy as np
import re
import time

categories = [
    "sci.med",
    "sci.space",
    "rec.sport.hockey",
    "talk.politics.guns",
]

train = fetch_20newsgroups(subset="train", categories=categories,
                           remove=("headers", "footers", "quotes"))
test  = fetch_20newsgroups(subset="test",  categories=categories,
                           remove=("headers", "footers", "quotes"))

print(f"Training documents : {len(train.data)}")
print(f"Test documents     : {len(test.data)}")
print(f"Categories         : {train.target_names}")
print()
for i, cat in enumerate(train.target_names):
    n = sum(train.target == i)
    print(f"  {cat:<30} {n} training docs")

In [ ]:
# Peek at one document
print(f"Category: {train.target_names[train.target[0]]}")
print()
print(train.data[0][:500])

### Method 1: BoW + Multinomial Naive Bayes

In [ ]:
t0 = time.time()

bow_vec = CountVectorizer(stop_words="english", min_df=2)
X_train_bow = bow_vec.fit_transform(train.data)
X_test_bow  = bow_vec.transform(test.data)

nb = MultinomialNB()
nb.fit(X_train_bow, train.target)
pred_bow = nb.predict(X_test_bow)

acc_bow = accuracy_score(test.target, pred_bow)
time_bow = time.time() - t0

print(f"BoW + Naive Bayes")
print(f"  Accuracy : {acc_bow:.3f}")
print(f"  Time     : {time_bow:.2f}s")
print(f"  Features : {X_train_bow.shape[1]:,} dimensions")

### Method 2: TF-IDF + Logistic Regression

In [ ]:
t0 = time.time()

tfidf_vec = TfidfVectorizer(stop_words="english", min_df=2, sublinear_tf=True)
X_train_tfidf = tfidf_vec.fit_transform(train.data)
X_test_tfidf  = tfidf_vec.transform(test.data)

lr = LogisticRegression(max_iter=1000, C=5)
lr.fit(X_train_tfidf, train.target)
pred_tfidf = lr.predict(X_test_tfidf)

acc_tfidf = accuracy_score(test.target, pred_tfidf)
time_tfidf = time.time() - t0

print(f"TF-IDF + Logistic Regression")
print(f"  Accuracy : {acc_tfidf:.3f}")
print(f"  Time     : {time_tfidf:.2f}s")
print(f"  Features : {X_train_tfidf.shape[1]:,} dimensions")

### Method 3: BM25 + weighted nearest-neighbor vote

For each test document, compute BM25 similarity to every training document.
Sum scores per class and pick the highest — a retrieval-based classifier.

In [ ]:
t0 = time.time()

import nltk
from nltk.corpus import stopwords as nltk_sw
nltk.download("stopwords", quiet=True)
sw = set(nltk_sw.words("english"))

def tokenize_for_bm25(text):
    tokens = re.sub(r"[^\w\s]", " ", text.lower()).split()
    return [t for t in tokens if t not in sw and len(t) > 1]

tokenized_train = [tokenize_for_bm25(doc) for doc in train.data]
tokenized_test  = [tokenize_for_bm25(doc) for doc in test.data]

bm25_clf = BM25Okapi(tokenized_train)
n_classes = len(categories)

pred_bm25 = []
for query_tokens in tokenized_test:
    scores = bm25_clf.get_scores(query_tokens)       # score vs all train docs
    class_scores = np.zeros(n_classes)
    for doc_idx, score in enumerate(scores):         # sum scores per class
        class_scores[train.target[doc_idx]] += score
    pred_bm25.append(int(np.argmax(class_scores)))

acc_bm25 = accuracy_score(test.target, pred_bm25)
time_bm25 = time.time() - t0

print(f"BM25 + nearest-neighbor vote")
print(f"  Accuracy : {acc_bm25:.3f}")
print(f"  Time     : {time_bm25:.2f}s")
print(f"  No feature matrix — retrieval-based")

### Results comparison

In [ ]:
print(f"Method                              Accuracy     Time")
print("-" * 55)
print(f"BoW + Naive Bayes                  {acc_bow:.3f}    {time_bow:.2f}s")
print(f"TF-IDF + Logistic Regression       {acc_tfidf:.3f}    {time_tfidf:.2f}s")
print(f"BM25 + NN vote                     {acc_bm25:.3f}    {time_bm25:.2f}s")

In [ ]:
# Per-class breakdown
import pandas as pd
short_names = ["sci.med", "sci.space", "hockey", "politics.guns"]

for method_name, preds in [("BoW+NB", pred_bow),
                            ("TF-IDF+LR", pred_tfidf),
                            ("BM25+NN", pred_bm25)]:
    report = classification_report(test.target, preds,
                                   target_names=short_names,
                                   output_dict=True)
    print(f"\n--- {method_name} ---")
    for cat in short_names:
        r = report[cat]
        print(f"  {cat:<18} precision={r['precision']:.2f}  recall={r['recall']:.2f}  f1={r['f1-score']:.2f}")

In [ ]:
import matplotlib.pyplot as plt

methods  = ["BoW+NB", "TF-IDF+LR", "BM25+NN"]
all_preds = [pred_bow, pred_tfidf, pred_bm25]
colors   = ["#4C72B0", "#DD8452", "#55A868"]

f1_data = {}
for name, preds in zip(methods, all_preds):
    report = classification_report(test.target, preds,
                                   target_names=short_names, output_dict=True)
    f1_data[name] = [report[c]["f1-score"] for c in short_names]

x = np.arange(len(short_names))
width = 0.25

fig, ax = plt.subplots(figsize=(10, 5))
for i, (method, color) in enumerate(zip(methods, colors)):
    ax.bar(x + i * width, f1_data[method], width, label=method, color=color, alpha=0.85)

ax.set_xlabel("Category", fontsize=12)
ax.set_ylabel("F1 score", fontsize=12)
ax.set_title("F1 per category: BoW vs TF-IDF vs BM25", fontsize=13)
ax.set_xticks(x + width)
ax.set_xticklabels(short_names, fontsize=11)
ax.set_ylim(0, 1.05)
ax.legend(fontsize=11)
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Top discriminative words per category (TF-IDF + LR coefficients)
feature_names = tfidf_vec.get_feature_names_out()

print("Top TF-IDF features per category (by LR coefficient):")
for class_idx, cat in enumerate(short_names):
    coef = lr.coef_[class_idx]
    top_idx = np.argsort(coef)[-10:][::-1]
    top_words = [f"{feature_names[i]}({coef[i]:.2f})" for i in top_idx]
    print(f"  {cat:<18} {', '.join(top_words)}")

In [ ]:
# A misclassified example — what fooled the model?
errors = [(i, test.target[i], pred_tfidf[i])
          for i in range(len(test.data))
          if test.target[i] != pred_tfidf[i]]

print(f"Total TF-IDF errors: {len(errors)} / {len(test.data)} ({len(errors)/len(test.data):.1%})")
print()
if errors:
    idx, true_label, pred_label = errors[0]
    print(f"Example misclassification:")
    print(f"  True      : {short_names[true_label]}")
    print(f"  Predicted : {short_names[pred_label]}")
    print(f"\nDocument (first 400 chars):")
    print(test.data[idx][:400])

### Key takeaways from the comparison

- **BoW + Naive Bayes** is fast and strong on topic classification — raw counts work because category words are distinctive by nature
- **TF-IDF + LR** outperforms BoW because IDF downweights common cross-category words and amplifies rare but discriminative ones
- **BM25 + NN** is a retrieval-based classifier with no training step — it underperforms here because BM25 was designed for **ranking**, not classification

> **Rule of thumb:**
> - Classification → TF-IDF features + trained classifier
> - Retrieval / RAG → BM25 as the lexical baseline
>
> Modern RAG systems often combine both: BM25 retrieves candidates, a neural re-ranker refines the order.

---
## Part 8 — Sentiment analysis: BoW vs TF-IDF vs BM25

**Task:** Twitter sentiment — classify tweets as positive or negative.

**Dataset:** NLTK `twitter_samples` — 5,000 positive + 5,000 negative tweets, balanced.

This task is deliberately different from topic classification:
- Topic classification is driven by **rare, distinctive nouns** (space, surgery, hockey)
- Sentiment is driven by **common opinion words** (good, bad, love, hate) that appear across all documents
- TF-IDF's IDF penalty may actually *hurt* here — sentiment words are frequent everywhere

We also add **Twitter-specific preprocessing**: removing @mentions, URLs, and handling hashtags.

In [ ]:
import nltk
import numpy as np
import re
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report
from rank_bm25 import BM25Okapi
import time

nltk.download("twitter_samples", quiet=True)
nltk.download("stopwords", quiet=True)
from nltk.corpus import twitter_samples
from nltk.corpus import stopwords as nltk_sw

pos_tweets = twitter_samples.strings("positive_tweets.json")
neg_tweets = twitter_samples.strings("negative_tweets.json")

texts  = pos_tweets + neg_tweets
labels = [1] * len(pos_tweets) + [0] * len(neg_tweets)

print(f"Total tweets  : {len(texts)}")
print(f"Positive      : {sum(labels)}")
print(f"Negative      : {len(labels) - sum(labels)}")
print()
print("Sample positive:", pos_tweets[0])
print("Sample negative:", neg_tweets[0])

### Twitter-specific preprocessing

In [ ]:
def preprocess_tweet(text):
    """Twitter-specific cleaning pipeline."""
    text = re.sub(r"http\S+|www\.\S+", "", text)      # remove URLs
    text = re.sub(r"@\w+", "", text)                   # remove @mentions
    text = re.sub(r"#(\w+)", r"\1", text)             # keep hashtag text, drop #
    text = re.sub(r"[^\w\s]", " ", text)              # remove punctuation
    text = re.sub(r"\s+", " ", text).strip().lower()   # normalize whitespace
    return text

examples = [
    "#FollowFriday @France_Inte @PKuchly57 for being top members :)",
    "I HATE this https://t.co/abc123 so much!!! it's the worst",
    "@someone feeling great today! #happy #blessed",
]
print("Before -> After preprocessing:")
for tweet in examples:
    print(f"  IN : {tweet}")
    print(f"  OUT: {preprocess_tweet(tweet)}")
    print()

cleaned_texts = [preprocess_tweet(t) for t in texts]

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    cleaned_texts, labels, test_size=0.2, random_state=42, stratify=labels
)
print(f"Training: {len(X_train)} tweets")
print(f"Test    : {len(X_test)} tweets")
print(f"Balance in test -- positive: {sum(y_test)}, negative: {len(y_test)-sum(y_test)}")

### Method 1: BoW + Multinomial Naive Bayes

In [ ]:
t0 = time.time()
bow_vec_s = CountVectorizer(ngram_range=(1, 2), min_df=3)
X_train_bow_s = bow_vec_s.fit_transform(X_train)
X_test_bow_s  = bow_vec_s.transform(X_test)
nb_s = MultinomialNB()
nb_s.fit(X_train_bow_s, y_train)
pred_bow_s = nb_s.predict(X_test_bow_s)
acc_bow_s  = accuracy_score(y_test, pred_bow_s)
time_bow_s = time.time() - t0
print(f"BoW (1+2-gram) + Naive Bayes")
print(f"  Accuracy : {acc_bow_s:.3f}  |  Time: {time_bow_s:.2f}s  |  Features: {X_train_bow_s.shape[1]:,}")
print()
print(classification_report(y_test, pred_bow_s, target_names=["negative", "positive"]))

### Method 2: TF-IDF + Logistic Regression

In [ ]:
t0 = time.time()
# sublinear_tf=False: for sentiment, word repetition IS signal
tfidf_vec_s = TfidfVectorizer(ngram_range=(1, 2), min_df=3, sublinear_tf=False)
X_train_tfidf_s = tfidf_vec_s.fit_transform(X_train)
X_test_tfidf_s  = tfidf_vec_s.transform(X_test)
lr_s = LogisticRegression(max_iter=1000, C=5)
lr_s.fit(X_train_tfidf_s, y_train)
pred_tfidf_s = lr_s.predict(X_test_tfidf_s)
acc_tfidf_s  = accuracy_score(y_test, pred_tfidf_s)
time_tfidf_s = time.time() - t0
print(f"TF-IDF (1+2-gram) + Logistic Regression")
print(f"  Accuracy : {acc_tfidf_s:.3f}  |  Time: {time_tfidf_s:.2f}s  |  Features: {X_train_tfidf_s.shape[1]:,}")
print()
print(classification_report(y_test, pred_tfidf_s, target_names=["negative", "positive"]))

### Method 3: BM25 + weighted nearest-neighbor vote

In [ ]:
t0 = time.time()
sw = set(nltk_sw.words("english"))

def tokenize_tweet(text):
    return [t for t in text.split() if t not in sw and len(t) > 1]

tok_train = [tokenize_tweet(t) for t in X_train]
tok_test  = [tokenize_tweet(t) for t in X_test]
y_train_arr = np.array(y_train)

bm25_s = BM25Okapi(tok_train)
pred_bm25_s = []
for query_tokens in tok_test:
    scores = bm25_s.get_scores(query_tokens)
    class_scores = np.zeros(2)
    for doc_idx, score in enumerate(scores):
        class_scores[y_train_arr[doc_idx]] += score
    pred_bm25_s.append(int(np.argmax(class_scores)))

acc_bm25_s  = accuracy_score(y_test, pred_bm25_s)
time_bm25_s = time.time() - t0
print(f"BM25 + nearest-neighbor vote")
print(f"  Accuracy : {acc_bm25_s:.3f}  |  Time: {time_bm25_s:.2f}s")
print()
print(classification_report(y_test, pred_bm25_s, target_names=["negative", "positive"]))

### Results summary and comparison with topic classification

In [ ]:
import matplotlib.pyplot as plt

methods_labels = ["BoW+NB", "TF-IDF+LR", "BM25+NN"]
acc_sentiment  = [acc_bow_s, acc_tfidf_s, acc_bm25_s]
acc_topic      = [acc_bow,   acc_tfidf,   acc_bm25]

x = np.arange(len(methods_labels))
width = 0.35

fig, ax = plt.subplots(figsize=(9, 5))
bars1 = ax.bar(x - width/2, acc_topic,     width, label="Topic (20 newsgroups)", color="#4C72B0", alpha=0.85)
bars2 = ax.bar(x + width/2, acc_sentiment, width, label="Sentiment (Twitter)",   color="#DD8452", alpha=0.85)

ax.set_ylabel("Accuracy", fontsize=12)
ax.set_title("Topic classification vs sentiment analysis", fontsize=13)
ax.set_xticks(x)
ax.set_xticklabels(methods_labels, fontsize=12)
ax.set_ylim(0.5, 1.05)
ax.legend(fontsize=11)
ax.grid(axis="y", alpha=0.3)
for bar in list(bars1) + list(bars2):
    ax.annotate(f"{bar.get_height():.3f}",
                (bar.get_x() + bar.get_width() / 2, bar.get_height()),
                ha="center", va="bottom", fontsize=10)
plt.tight_layout()
plt.show()

In [ ]:
# Top positive and negative words learned by the model
feature_names_s = tfidf_vec_s.get_feature_names_out()
coef = lr_s.coef_[0]

top_pos_idx = np.argsort(coef)[-15:][::-1]
top_neg_idx = np.argsort(coef)[:15]

print("Top POSITIVE features (TF-IDF+LR coefficients):")
for i in top_pos_idx:
    print(f"  {feature_names_s[i]:<25} {coef[i]:+.3f}")

print()
print("Top NEGATIVE features (TF-IDF+LR coefficients):")
for i in top_neg_idx:
    print(f"  {feature_names_s[i]:<25} {coef[i]:+.3f}")

In [ ]:
# Negation challenge: does the model handle "not good"?
negation_tweets = [
    "i do not like this at all",
    "this is not good",
    "not bad actually pretty good",
    "i love it so much",
    "i hate everything today",
]

print("Negation and sentiment predictions (TF-IDF+LR):")
print(f"  {'Tweet':<40} Predicted")
print("-" * 60)
for tweet in negation_tweets:
    cleaned = preprocess_tweet(tweet)
    vec = tfidf_vec_s.transform([cleaned])
    pred = lr_s.predict(vec)[0]
    label = "POSITIVE" if pred == 1 else "NEGATIVE"
    print(f"  {tweet:<40} {label}")

In [ ]:
# Are negation bigrams in the vocabulary and correctly signed?
negation_bigrams = ["not good", "not bad", "not like", "not great", "not happy", "so good", "so bad"]
print("Negation bigrams in vocabulary:")
for bigram in negation_bigrams:
    if bigram in tfidf_vec_s.vocabulary_:
        idx = tfidf_vec_s.vocabulary_[bigram]
        sign = "positive" if coef[idx] > 0 else "negative"
        print(f"  '{bigram}': coeff={coef[idx]:+.3f} -> {sign}")
    else:
        print(f"  '{bigram}': NOT in vocabulary (too rare in training data)")

In [ ]:
# IDF scores: why sentiment words are penalized
idf_lookup = dict(zip(tfidf_vec_s.get_feature_names_out(), tfidf_vec_s.idf_))

sentiment_words = ["love", "hate", "great", "terrible", "happy", "sad", "good", "bad", "awesome", "horrible"]
print("IDF of core sentiment words (lower = more common = penalized more):")
for word in sentiment_words:
    if word in idf_lookup:
        print(f"  {word:<12} IDF = {idf_lookup[word]:.3f}")

print()
# Compare to topic-specific words
topic_words_in_tweets = [w for w in ["nasa", "surgery", "hockey", "gun", "space"] if w in idf_lookup]
print("IDF of topic-specific words for comparison:")
for word in topic_words_in_tweets:
    print(f"  {word:<12} IDF = {idf_lookup[word]:.3f}")

print()
print("=> Sentiment words have LOW IDF (common everywhere) -> TF-IDF downweights exactly the words that matter most")

### Key takeaways

**Why sentiment is harder than topic classification for these methods:**
- Sentiment words ("good", "bad", "love", "hate") appear in **both** positive and negative documents
  — their IDF is low, so TF-IDF penalizes the words that matter most
- Topic words ("surgery", "nasa", "hockey") are rare outside their category
  — IDF amplifies them, which is exactly what we want

**Bigrams help, but only partially:**
- `"not good"` as a bigram captures negation that separate unigrams cannot
- But rare negation patterns may not appear enough times in training to make it into the vocabulary (`min_df=3`)

**BM25 on sentiment:**
- TF saturation hurts: repeated emotional words ("love love love") carry genuine emphasis in tweets
- Retrieval-based classification struggles because tweets from the same sentiment class are topically diverse

> **Bottom line:** for sentiment analysis, classical methods top out around 75-80% on Twitter data.  
> The real leap comes from **transformer models** (BERT, RoBERTa, BERTweet) that understand  
> context, negation, and sarcasm through attention — which is the topic of a future lecture.

---
## Summary

We covered the full low-level NLP preprocessing pipeline:

| Step | Tools |
|---|---|
| Cleaning & normalization | `re`, `unicodedata`, `BeautifulSoup` |
| Stop words | `nltk.corpus.stopwords` |
| Stemming | `nltk.stem.PorterStemmer`, `SnowballStemmer` |
| Lemmatization | `spacy` |
| N-grams | `nltk.ngrams` |
| Word tokenization | `nltk.word_tokenize`, `sent_tokenize` |
| Subword tokenization | `transformers` (GPT-2 BPE, BERT WordPiece, T5 SentencePiece) |
| Bag of words | `sklearn.CountVectorizer` |
| TF-IDF | `sklearn.TfidfVectorizer` |
| BM25 | `rank_bm25.BM25Okapi` |

**Next lecture:** Word embeddings — dense, semantic vector representations that solve the sparsity and semantics limitations we saw here.